# Regional Inequality and the Kuznets Curve: Panel Fixed Effects in Python

Does economic growth reduce inequality within countries, or does it make some regions richer while others fall behind? Using satellite nighttime light data from **180 countries (1990--2013)**, Lessmann and Seidel (2017) found that the relationship is **N-shaped** -- not an inverted-U. Inequality rises at low income levels, falls through middle-income development, then rises *again* at very high incomes.

In this notebook we replicate their key findings using [PyFixest](https://pyfixest.org/) for panel fixed effects estimation and [Great Tables](https://posit-dev.github.io/great-tables/) for publication-quality regression tables.

**Learning objectives:**

- Understand why polynomial specifications are necessary for testing the Kuznets hypothesis
- Implement pooled OLS and two-way fixed effects regressions using PyFixest
- Compute and interpret turning points of a cubic polynomial
- Compare pooled OLS and TWFE estimates to assess omitted variable bias
- Identify the key determinants of regional inequality

## Setup and imports

Install required packages (not available by default in Google Colab):

In [ ]:
!pip install pyfixest great_tables -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pyfixest as pf
from great_tables import GT, md, style, loc

# Reproducibility
np.random.seed(42)

# Color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

# Data URLs
URL_TAB03 = "https://github.com/quarcs-lab/data-open/raw/master/pGDP/simpleTAB03.dta"
URL_TAB04 = "https://github.com/quarcs-lab/data-open/raw/master/pGDP/simpleTAB04.dta"

# Significance stars helper
def stars(pval):
    if pval < 0.01: return "***"
    elif pval < 0.05: return "**"
    elif pval < 0.10: return "*"
    return ""

## Data loading and panel structure

The dataset comes from Lessmann and Seidel (2017), who measured regional inequality using satellite nighttime light data. The dependent variable is a population-weighted **Gini coefficient** -- a number between 0 (perfect equality across regions) and 1 (all income in one region).

In [ ]:
df3 = pd.read_stata(URL_TAB03)
print(f"Shape: {df3.shape}")
print(f"Columns: {list(df3.columns)}")
print(f"\nDescriptive statistics:")
print(df3.describe().round(4))
print(f"\nPanel structure:")
print(f"  Countries: {df3['id'].nunique()}")
print(f"  Time periods: {sorted(df3['year'].unique())}")
print(f"\nObservations per period:")
print(df3.groupby('year')['id'].count())

The dataset has **880 observations** across 180 countries and 5 time periods (5-year averages: 1990--1994, 1995--1999, 2000--2004, 2005--2009, 2010--2013). The mean regional Gini is 0.064 (SD = 0.033). Log GDP per capita ranges from 5.25 (about \$190) to 11.67 (about \$117,000).

Now load the determinants dataset:

In [ ]:
df4 = pd.read_stata(URL_TAB04)
print(f"Shape: {df4.shape}")
print(f"\nKey variables: gini, lnGDPpc, rents, land, trade, fdi,")
print(f"               gasoline, areaXgasoline, aid, school, ethnic_gini")
print(f"\nNotable missing values:")
for var in ['aid', 'school', 'ethnic_gini']:
    n_obs = df4[var].notna().sum()
    pct_miss = df4[var].isna().mean()
    print(f"  {var:12s}: {n_obs} / 880 ({pct_miss:.0%} missing)")

## Visual exploration: Is there a Kuznets curve?

We plot every country-period observation with three polynomial fits overlaid. Think of a cubic polynomial as a **roller coaster track** -- it can climb, descend, and rise again, capturing patterns that a straight line would miss.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = df3["log_GDPpc"].values
y = df3["gini"].values

ax.scatter(x, y, alpha=0.35, s=18, color=STEEL_BLUE, edgecolors="white", linewidth=0.3)

x_grid = np.linspace(x.min(), x.max(), 200)
for deg, color, ls, lw, label in [
    (1, "gray", "--", 1.5, "Linear"),
    (2, TEAL, "--", 1.8, "Quadratic (inverted-U)"),
    (3, WARM_ORANGE, "-", 2.5, "Cubic (N-shape)"),
]:
    coeffs = np.polyfit(x, y, deg)
    ax.plot(x_grid, np.polyval(coeffs, x_grid), color=color, ls=ls, lw=lw, label=label)

ax.set_xlabel("Log GDP per capita (PPP, constant US$)")
ax.set_ylabel("Regional Inequality (Population-weighted Gini)")
ax.set_title("Regional Inequality vs National Development\n180 Countries, 1990-2013 (pooled)")
ax.legend()
plt.tight_layout()
plt.show()

The cubic (N-shaped) fit captures the data better than the linear or quadratic. Is this pattern stable across time periods?

In [ ]:
periods = sorted(df3["year"].unique())
period_labels = {1: "1990-1994", 2: "1995-1999", 3: "2000-2004",
                 4: "2005-2009", 5: "2010-2013"}

fig, axes = plt.subplots(1, len(periods), figsize=(20, 5), sharey=True)
for ax, period in zip(axes, periods):
    sub = df3[df3["year"] == period]
    ax.scatter(sub["log_GDPpc"], sub["gini"], alpha=0.4, s=20, color=STEEL_BLUE)
    cp = np.polyfit(sub["log_GDPpc"], sub["gini"], 3)
    xg = np.linspace(sub["log_GDPpc"].min(), sub["log_GDPpc"].max(), 100)
    ax.plot(xg, np.polyval(cp, xg), color=WARM_ORANGE, lw=2)
    ax.set_title(period_labels.get(int(period), f"Period {int(period)}"))
    ax.set_xlabel("Log GDP pc")
axes[0].set_ylabel("Regional Gini")
fig.suptitle("Inequality-Development Relationship by Period", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

The N-shape appears in all five periods -- a stable structural relationship.

## Pooled OLS baseline: Linear, quadratic, and cubic

The cubic polynomial specification is:

$$\text{Gini}_i = \beta_0 + \beta_1 \ln(\text{GDP}_i) + \beta_2 [\ln(\text{GDP}_i)]^2 + \beta_3 [\ln(\text{GDP}_i)]^3 + \epsilon_i$$

where $\beta_1$ captures the linear association, $\beta_2$ allows the relationship to bend once, and $\beta_3$ allows it to bend a second time (N-shape if positive). These correspond to `log_GDPpc`, `log_GDPpc2`, and `log_GDPpc3` in the code.

We cluster standard errors at the country level to account for serial correlation:

In [ ]:
ols_linear = pf.feols("gini ~ log_GDPpc", data=df3, vcov={"CRV1": "id"})
ols_quad = pf.feols("gini ~ log_GDPpc + log_GDPpc2", data=df3, vcov={"CRV1": "id"})
ols_cubic = pf.feols("gini ~ log_GDPpc + log_GDPpc2 + log_GDPpc3", data=df3,
                      vcov={"CRV1": "id"})

print("Pooled OLS Coefficient Comparison:")
print(f"{'Variable':<14} {'Linear':>10} {'Quadratic':>12} {'Cubic':>10}")
print("-" * 48)
for var in ["log_GDPpc", "log_GDPpc2", "log_GDPpc3"]:
    vals = []
    for m in [ols_linear, ols_quad, ols_cubic]:
        vals.append(f"{m.coef()[var]:.4f}" if var in m.coef().index else "---")
    print(f"{var:<14} {vals[0]:>10} {vals[1]:>12} {vals[2]:>10}")

The cubic specification reveals the N-shaped pattern (coefficients: 0.241, -0.028, 0.001) but all terms are only *marginally* significant (p ~ 0.07--0.09). Pooled OLS confounds between-country and within-country variation.

## Why fixed effects? The omitted variable problem

Countries differ in geography, institutions, and culture -- factors that affect *both* inequality *and* development. Think of it like measuring whether nutrition affects height: you can't just compare children from different families (taller families eat differently). You need to look at changes *within the same family*. **Fixed effects** does this for countries.

The spaghetti plot below shows individual country trajectories vs the pooled pattern:

In [ ]:
# Select 20 countries across the GDP distribution
country_obs = df3.groupby("id").agg(
    n_periods=("year", "count"), mean_gdp=("log_GDPpc", "mean")
).reset_index()
country_obs = country_obs[country_obs["n_periods"] >= 3].sort_values("mean_gdp")
idx = np.linspace(0, len(country_obs) - 1, 20, dtype=int)
selected_ids = country_obs.iloc[idx]["id"].values

fig, ax = plt.subplots(figsize=(10, 6))
for cid in selected_ids:
    sub = df3[df3["id"] == cid].sort_values("log_GDPpc")
    ax.plot(sub["log_GDPpc"], sub["gini"], color="gray", alpha=0.3,
            lw=1.2, marker="o", ms=3)

# Highlight 6 diverse countries
highlight_ids = country_obs.iloc[
    np.linspace(0, len(country_obs) - 1, 6, dtype=int)
]["id"].values
colors = [WARM_ORANGE, TEAL, STEEL_BLUE, "#e8956a", "#8ec8e8", "#66c8b8"]
for i, cid in enumerate(highlight_ids):
    sub = df3[df3["id"] == cid].sort_values("log_GDPpc")
    ax.plot(sub["log_GDPpc"], sub["gini"], color=colors[i], lw=2.5,
            marker="o", ms=5, label=sub["country"].iloc[0])

# Pooled cubic fit
c3 = np.polyfit(df3["log_GDPpc"].values, df3["gini"].values, 3)
ax.plot(x_grid, np.polyval(c3, x_grid), color=WARM_ORANGE, ls="--", lw=2,
        alpha=0.5, label="Pooled cubic fit")

ax.set_xlabel("Log GDP per capita")
ax.set_ylabel("Regional Gini")
ax.set_title("Individual Country Trajectories vs Pooled Pattern")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

Individual countries follow trajectories that differ from the pooled cross-sectional pattern. Fixed effects remove country-specific levels and focus only on *within-country* changes.

## Two-way fixed effects: Replicating Table 3

Two-way fixed effects (TWFE) adds country effects $\alpha_i$ and year effects $\gamma_t$:

$$\text{Gini}_{it} = \beta_1 \ln(\text{GDP}_{it}) + \beta_2 [\ln(\text{GDP}_{it})]^2 + \beta_3 [\ln(\text{GDP}_{it})]^3 + \alpha_i + \gamma_t + \epsilon_{it}$$

In PyFixest, fixed effects go after a pipe `|`: `gini ~ log_GDPpc | id + year`.

In [ ]:
fe_linear = pf.feols("gini ~ log_GDPpc | id + year", data=df3, vcov={"CRV1": "id"})
fe_quad = pf.feols("gini ~ log_GDPpc + log_GDPpc2 | id + year", data=df3,
                    vcov={"CRV1": "id"})
fe_cubic = pf.feols("gini ~ log_GDPpc + log_GDPpc2 + log_GDPpc3 | id + year",
                     data=df3, vcov={"CRV1": "id"})

print("TWFE Cubic Model:")
print(fe_cubic.summary())

All three terms are highly significant (p < 0.001), confirming the N-shape *within countries*. The overall R² of 0.975 means country FE absorb most variation. The within-R² of 0.142 shows the cubic explains 14.2% of within-country variation -- substantial for 5 periods.

### Publication-quality Table 3 (Great Tables)

In [ ]:
fe_models = [fe_linear, fe_quad, fe_cubic]
fe_model_names = ["(1) Gini", "(2) Gini", "(3) Gini"]
fe_vars = ["log_GDPpc", "log_GDPpc2", "log_GDPpc3"]
fe_var_labels = {"log_GDPpc": "ln(GDP pc)", "log_GDPpc2": "ln(GDP pc)\u00b2",
                 "log_GDPpc3": "ln(GDP pc)\u00b3"}

table3_rows = []
for var in fe_vars:
    row_coef = {"Variable": fe_var_labels[var]}
    row_se = {"Variable": ""}
    for i, model in enumerate(fe_models):
        col = fe_model_names[i]
        tidy = model.tidy()
        if var in tidy.index:
            c = tidy.loc[var, "Estimate"]
            se = tidy.loc[var, "Std. Error"]
            p = tidy.loc[var, "Pr(>|t|)"]
            row_coef[col] = f"{c:.3f}{stars(p)}"
            row_se[col] = f"({se:.3f})"
        else:
            row_coef[col] = ""
            row_se[col] = ""
    table3_rows.append(row_coef)
    table3_rows.append(row_se)

table3_rows.append({k: "" for k in ["Variable"] + fe_model_names})
table3_rows.append({"Variable": "Observations",
                     "(1) Gini": str(fe_linear._N),
                     "(2) Gini": str(fe_quad._N),
                     "(3) Gini": str(fe_cubic._N)})
table3_rows.append({"Variable": "R-squared",
                     "(1) Gini": f"{fe_linear._r2:.3f}",
                     "(2) Gini": f"{fe_quad._r2:.3f}",
                     "(3) Gini": f"{fe_cubic._r2:.3f}"})
table3_rows.append({"Variable": "Country FE",
                     "(1) Gini": "Yes", "(2) Gini": "Yes", "(3) Gini": "Yes"})
table3_rows.append({"Variable": "Year FE",
                     "(1) Gini": "Yes", "(2) Gini": "Yes", "(3) Gini": "Yes"})

table3_df = pd.DataFrame(table3_rows)
(
    GT(table3_df)
    .tab_header(
        title=md("**Table 3: Regional Inequality and Development (Kuznets Curve)**"),
        subtitle="Dependent variable: Population-weighted regional Gini index"
    )
    .tab_source_note(
        "Notes: Two-way fixed effects (country + year). "
        "Robust standard errors clustered at country level in parentheses. "
        "* p<0.10, ** p<0.05, *** p<0.01."
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.body(columns="Variable"))
)

### The linear TWFE model is uninformative

A researcher who only estimates the linear specification would miss the relationship entirely:

In [ ]:
print("Linear TWFE:")
print(f"  log_GDPpc: {fe_linear.coef()['log_GDPpc']:.3f} "
      f"(SE {fe_linear.se()['log_GDPpc']:.3f}, "
      f"p = {fe_linear.pvalue()['log_GDPpc']:.3f})")
print(f"  R-squared Within: {fe_linear._r2_within:.3f}")

The linear coefficient is -0.003 (p = 0.265) with within-R² of only 0.009. The opposing effects at different development stages cancel out.

## The N-shaped curve: Computing turning points

To find where the curve bends, set the first derivative to zero:

$$\frac{\partial \, \text{Gini}}{\partial \ln(\text{GDP})} = \beta_1 + 2\beta_2 \ln(\text{GDP}) + 3\beta_3 [\ln(\text{GDP})]^2 = 0$$

In [ ]:
b1 = fe_cubic.coef()["log_GDPpc"]
b2 = fe_cubic.coef()["log_GDPpc2"]
b3 = fe_cubic.coef()["log_GDPpc3"]

roots = np.roots([3 * b3, 2 * b2, b1])
real_roots = np.sort(roots[np.isreal(roots)].real)
turning_usd = np.exp(real_roots)

print(f"Cubic TWFE coefficients: b1 = {b1:.6f}, b2 = {b2:.6f}, b3 = {b3:.6f}")
print(f"Turning points (log scale): [{real_roots[0]:.3f}, {real_roots[1]:.3f}]")
print(f"Turning points (USD PPP):   [${turning_usd[0]:,.0f}, ${turning_usd[1]:,.0f}]")
print(f"\nThree development phases:")
print(f"  Below ${turning_usd[0]:,.0f}: inequality RISES (initial concentration)")
print(f"  ${turning_usd[0]:,.0f} to ${turning_usd[1]:,.0f}: inequality FALLS (convergence)")
print(f"  Above ${turning_usd[1]:,.0f}: inequality may RISE again (knowledge economy)")

In [ ]:
# Fitted N-shaped curve with turning points
tp1_log, tp2_log = real_roots[0], real_roots[1]
tp1_usd, tp2_usd = turning_usd[0], turning_usd[1]

x_min, x_max = df3["log_GDPpc"].min(), df3["log_GDPpc"].max()
x_fit = np.linspace(x_min, x_max, 300)
y_fit = b1 * x_fit + b2 * x_fit**2 + b3 * x_fit**3
y_offset = df3["gini"].mean() - np.mean(y_fit)
y_fit_adj = y_fit + y_offset

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_fit, y_fit_adj, color=WARM_ORANGE, lw=3, label="Fitted cubic polynomial")

# Shade the three regions
y_bottom = y_fit_adj.min() - 0.005
mask_rise1 = x_fit <= tp1_log
ax.fill_between(x_fit[mask_rise1], y_fit_adj[mask_rise1], y_bottom,
                alpha=0.15, color=WARM_ORANGE, label="Rising inequality")
mask_fall = (x_fit >= tp1_log) & (x_fit <= min(tp2_log, x_max))
ax.fill_between(x_fit[mask_fall], y_fit_adj[mask_fall], y_bottom,
                alpha=0.15, color=STEEL_BLUE, label="Falling inequality")
if tp2_log < x_max:
    mask_rise2 = x_fit >= tp2_log
    ax.fill_between(x_fit[mask_rise2], y_fit_adj[mask_rise2], y_bottom,
                    alpha=0.15, color=WARM_ORANGE)

# Turning point markers
ax.axvline(tp1_log, color=TEAL, ls="--", lw=1.5, alpha=0.8)
ax.axvline(tp2_log, color=TEAL, ls="--", lw=1.5, alpha=0.8)

y_range = y_fit_adj.max() - y_fit_adj.min()
ax.annotate(f"Peak: ${tp1_usd:,.0f}",
            xy=(tp1_log, y_fit_adj[mask_rise1][-1]),
            xytext=(tp1_log + 0.3, y_fit_adj.max() - 0.1 * y_range),
            fontsize=11, color=TEAL, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.5))
ax.annotate(f"Trough: ${tp2_usd:,.0f}",
            xy=(min(tp2_log, x_max), y_fit_adj[np.argmin(np.abs(x_fit - tp2_log))]),
            xytext=(min(tp2_log, x_max) - 1.5, y_fit_adj.min() + 0.1 * y_range),
            fontsize=11, color=TEAL, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.5))

# Secondary USD x-axis
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
usd_ticks = [5, 6, 7, 8, 9, 10, 11]
ax2.set_xticks(usd_ticks)
ax2.set_xticklabels([f"${np.exp(v):,.0f}" for v in usd_ticks], fontsize=9)
ax2.set_xlabel("GDP per capita (PPP, US$)")

ax.set_xlabel("Log GDP per capita")
ax.set_ylabel("Predicted Regional Gini")
ax.set_title("The N-Shaped Kuznets Curve\nRegional Inequality and Development")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

The peak at \$2,287 and trough at \$77,205 define three development phases. These closely replicate the paper's values (\$2,288 and \$77,128).

## Pooled OLS vs TWFE: Correcting for omitted variable bias

In [ ]:
print("Pooled OLS vs TWFE (cubic):")
print(f"{'Variable':<14} {'Pooled OLS':>12} {'TWFE':>12}")
print("-" * 40)
for var in ["log_GDPpc", "log_GDPpc2", "log_GDPpc3"]:
    print(f"{var:<14} {ols_cubic.coef()[var]:>12.4f} {fe_cubic.coef()[var]:>12.4f}")

# Coefficient comparison plot
fig, ax = plt.subplots(figsize=(9, 5))
var_names = ["log_GDPpc", "log_GDPpc2", "log_GDPpc3"]
var_display = ["ln(GDP pc)", "ln(GDP pc)\u00b2", "ln(GDP pc)\u00b3"]
y_positions = np.arange(len(var_names))
bar_height = 0.35

for i, var in enumerate(var_names):
    ols_tidy = ols_cubic.tidy()
    ax.barh(y_positions[i] + bar_height/2, ols_tidy.loc[var, "Estimate"],
            height=bar_height, color=STEEL_BLUE, alpha=0.8,
            label="Pooled OLS" if i == 0 else None,
            xerr=1.96 * ols_tidy.loc[var, "Std. Error"], capsize=5,
            error_kw={"ecolor": NEAR_BLACK, "lw": 1.5, "capthick": 1.5})
    fe_tidy = fe_cubic.tidy()
    ax.barh(y_positions[i] - bar_height/2, fe_tidy.loc[var, "Estimate"],
            height=bar_height, color=WARM_ORANGE, alpha=0.8,
            label="TWFE" if i == 0 else None,
            xerr=1.96 * fe_tidy.loc[var, "Std. Error"], capsize=5,
            error_kw={"ecolor": NEAR_BLACK, "lw": 1.5, "capthick": 1.5})

ax.axvline(0, color="gray", ls="-", lw=0.8, alpha=0.5)
ax.set_yticks(y_positions)
ax.set_yticklabels(var_display)
ax.set_xlabel("Coefficient Estimate (with 95% CI)")
ax.set_title("Pooled OLS vs Two-Way Fixed Effects\nCubic Kuznets Specification")
ax.legend()
plt.tight_layout()
plt.show()

TWFE coefficients are larger and highly significant (p < 0.001) vs marginally significant (p ~ 0.07) for pooled OLS. Fixed effects both correct bias and improve precision.

## Determinants of regional inequality

### Correlation structure

In [ ]:
det_vars = ["gini", "lnGDPpc", "rents", "land", "trade", "fdi",
            "gasoline", "aid", "school", "ethnic_gini"]
det_labels = ["Gini", "ln(GDP pc)", "Resource\nrents", "Arable\nland",
              "Trade\nopenness", "FDI", "Gasoline\nprice", "Foreign\naid",
              "School\nenroll.", "Ethnic\nGini"]
corr = df4[det_vars].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(det_vars)))
ax.set_yticks(range(len(det_vars)))
ax.set_xticklabels(det_labels, fontsize=9, rotation=45, ha="right")
ax.set_yticklabels(det_labels, fontsize=9)
ax.grid(False)
for i in range(len(det_vars)):
    for j in range(len(det_vars)):
        val = corr.values[i, j]
        text_color = "white" if abs(val) > 0.4 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, fontweight="bold", color=text_color)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation Matrix: Determinants of Regional Inequality", pad=15)
plt.tight_layout()
plt.show()

Ethnic Gini has the strongest positive correlation with regional inequality (r = 0.49). School enrollment has the strongest negative (r = -0.41).

### Determinant regressions: Replicating Table 4

Five TWFE models, each adding a different group of determinants:

In [ ]:
det1 = pf.feols("gini ~ lnGDPpc + lnGDPpc2 + lnGDPpc3 + rents + land | id + year",
                data=df4, vcov={"CRV1": "id"})
det2 = pf.feols("gini ~ lnGDPpc + lnGDPpc2 + lnGDPpc3 + trade + fdi | id + year",
                data=df4, vcov={"CRV1": "id"})
det3 = pf.feols("gini ~ lnGDPpc + lnGDPpc2 + lnGDPpc3 + gasoline + areaXgasoline "
                "| id + year", data=df4, vcov={"CRV1": "id"})
det4 = pf.feols("gini ~ lnGDPpc + lnGDPpc2 + lnGDPpc3 + aid + school | id + year",
                data=df4, vcov={"CRV1": "id"})
det5 = pf.feols("gini ~ lnGDPpc + lnGDPpc2 + lnGDPpc3 + ethnic_gini | id + year",
                data=df4, vcov={"CRV1": "id"})

det_models = [det1, det2, det3, det4, det5]
det_names = ["Resources", "Trade", "Mobility", "Aid/Education", "Ethnicity"]
print("Determinant model summaries:")
for name, m in zip(det_names, det_models):
    print(f"\n--- {name} (N={m._N}) ---")
    print(m.summary())

### Publication-quality Table 4 (Great Tables)

In [ ]:
det_model_cols = ["(1) Gini", "(2) Gini", "(3) Gini", "(4) Gini", "(5) Gini"]
det_all_vars = ["lnGDPpc", "lnGDPpc2", "lnGDPpc3", "rents", "land",
                "trade", "fdi", "gasoline", "areaXgasoline", "aid",
                "school", "ethnic_gini"]
det_var_labels = {
    "lnGDPpc": "ln(GDP pc)", "lnGDPpc2": "ln(GDP pc)\u00b2",
    "lnGDPpc3": "ln(GDP pc)\u00b3", "rents": "Resource rents",
    "land": "Arable land", "trade": "Trade openness", "fdi": "FDI",
    "gasoline": "Gasoline price", "areaXgasoline": "Area \u00d7 Gasoline",
    "aid": "Foreign aid", "school": "School enrollment",
    "ethnic_gini": "Ethnic Gini",
}

table4_rows = []
for var in det_all_vars:
    row_coef = {"Variable": det_var_labels.get(var, var)}
    row_se = {"Variable": ""}
    for i, model in enumerate(det_models):
        col = det_model_cols[i]
        tidy = model.tidy()
        if var in tidy.index:
            c = tidy.loc[var, "Estimate"]
            se = tidy.loc[var, "Std. Error"]
            p = tidy.loc[var, "Pr(>|t|)"]
            row_coef[col] = f"{c:.3f}{stars(p)}"
            row_se[col] = f"({se:.3f})"
        else:
            row_coef[col] = ""
            row_se[col] = ""
    table4_rows.append(row_coef)
    table4_rows.append(row_se)

table4_rows.append({k: "" for k in ["Variable"] + det_model_cols})
for stat_name, stat_fn in [("Observations", lambda m: str(m._N)),
                            ("R-squared", lambda m: f"{m._r2:.3f}"),
                            ("Country FE", lambda m: "Yes"),
                            ("Year FE", lambda m: "Yes")]:
    row = {"Variable": stat_name}
    for i, model in enumerate(det_models):
        row[det_model_cols[i]] = stat_fn(model)
    table4_rows.append(row)

table4_df = pd.DataFrame(table4_rows)
(
    GT(table4_df)
    .tab_header(
        title=md("**Table 4: Determinants of Regional Inequality**"),
        subtitle="Dependent variable: Population-weighted regional Gini index"
    )
    .tab_source_note(
        "Notes: Two-way fixed effects (country + year). "
        "All models include cubic polynomial of ln(GDP pc). "
        "Robust standard errors clustered at country level. "
        "* p<0.10, ** p<0.05, *** p<0.01."
    )
    .tab_style(style=style.text(weight="bold"), locations=loc.body(columns="Variable"))
)

Seven of nine determinants are significant. **Ethnic income inequality** dominates (0.071, p < 0.001) -- 3.9x larger than the next biggest positive effect. Arable land (-0.053) and school enrollment (-0.014) are the only factors that *reduce* inequality.

### Coefficient stability across specifications

In [ ]:
all_models = [fe_cubic] + det_models
all_spec_names = ["Baseline\n(Table 3)", "Resources", "Trade", "Mobility",
                  "Aid/Educ.", "Ethnicity"]
poly_vars = ["lnGDPpc", "lnGDPpc2", "lnGDPpc3"]
poly_vars_tab3 = ["log_GDPpc", "log_GDPpc2", "log_GDPpc3"]
poly_labels = ["ln(GDP pc)", "ln(GDP pc)\u00b2", "ln(GDP pc)\u00b3"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
for j, (var4, var3, label) in enumerate(zip(poly_vars, poly_vars_tab3, poly_labels)):
    ax = axes[j]
    coefs, cis_lo, cis_hi = [], [], []
    for i, model in enumerate(all_models):
        tidy = model.tidy()
        v = var3 if i == 0 else var4
        if v in tidy.index:
            c = tidy.loc[v, "Estimate"]
            se = tidy.loc[v, "Std. Error"]
            coefs.append(c)
            cis_lo.append(c - 1.96 * se)
            cis_hi.append(c + 1.96 * se)
        else:
            coefs.append(np.nan); cis_lo.append(np.nan); cis_hi.append(np.nan)

    x_pos = np.arange(len(all_models))
    colors = [TEAL] + [STEEL_BLUE] * len(det_models)
    ax.scatter(x_pos, coefs, color=colors, s=60, zorder=5)
    for k in range(len(all_models)):
        ax.plot([x_pos[k], x_pos[k]], [cis_lo[k], cis_hi[k]], color=colors[k], lw=2)
    ax.axhline(0, color="gray", ls="-", lw=0.5, alpha=0.4)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(all_spec_names, fontsize=8, rotation=30, ha="right")
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("Coefficient" if j == 0 else "")

fig.suptitle("Stability of Kuznets Curve Coefficients Across Specifications",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

The sign pattern (+, -, +) is preserved across all six specifications. Magnitudes attenuate most when ethnic inequality is controlled (linear term drops from 0.293 to 0.149), suggesting part of the "development effect" is actually ethnic composition.

### Determinant effects at a glance

In [ ]:
det_only_vars = ["rents", "land", "trade", "fdi", "gasoline",
                 "areaXgasoline", "aid", "school", "ethnic_gini"]
det_only_labels = ["Resource rents", "Arable land", "Trade openness", "FDI",
                   "Gasoline price", "Area \u00d7 Gasoline",
                   "Foreign aid", "School enrollment", "Ethnic Gini"]

det_coefs, det_sigs, det_display = [], [], []
for var, label in zip(det_only_vars, det_only_labels):
    for model in det_models:
        tidy = model.tidy()
        if var in tidy.index:
            det_coefs.append(tidy.loc[var, "Estimate"])
            det_sigs.append(tidy.loc[var, "Pr(>|t|)"] < 0.10)
            det_display.append(label)
            break

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = [WARM_ORANGE if c > 0 else STEEL_BLUE for c in det_coefs]
alphas = [0.9 if sig else 0.4 for sig in det_sigs]
bars = ax.barh(range(len(det_display)), det_coefs, color=colors_bar, height=0.6)
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)
ax.axvline(0, color="gray", ls="-", lw=0.8, alpha=0.5)
ax.set_yticks(range(len(det_display)))
ax.set_yticklabels(det_display)
ax.set_xlabel("Coefficient (effect on regional Gini)")
ax.set_title("Determinants of Regional Inequality\n"
             "Solid = significant (p<0.10), faded = not significant")
ax.legend(handles=[
    mpatches.Patch(color=WARM_ORANGE, alpha=0.9, label="Increases inequality"),
    mpatches.Patch(color=STEEL_BLUE, alpha=0.9, label="Decreases inequality")
], loc="lower right")
plt.tight_layout()
plt.show()

## Discussion

**The relationship is N-shaped, not inverted-U.** The cubic TWFE yields coefficients 0.293, -0.032, 0.001 (all p < 0.001) with turning points at \$2,287 and \$77,205. The N-shape is robust across all six specifications.

**Fixed effects are essential.** The linear TWFE completely misses the relationship (p = 0.265). Pooled OLS cubic is only marginally significant (p ~ 0.07), while TWFE is highly significant (p < 0.001).

**Ethnic income inequality is the strongest determinant** (0.071, 3.9x larger than any other positive effect). Controlling for it halves the Kuznets curve, suggesting part of the development-inequality relationship reflects ethnic composition.

**Caveats:** The second turning point (\$77,205) relies on very few observations. Within-R-squared ranges from 0.01 to 0.28. All results are associations, not causal effects.

## Summary

1. **N-shaped Kuznets curve** confirmed with TWFE (within-R² = 0.142 vs 0.009 for linear)
2. **Turning points** at \$2,287 (peak inequality) and \$77,205 (trough), stable across periods
3. **Ethnic Gini (0.071)** is the dominant determinant -- 3.9x larger than the next biggest
4. **Both polynomial specification AND fixed effects** are needed to reveal the true pattern

## References

1. [Lessmann, C., & Seidel, A. (2017). Regional inequality, convergence, and its determinants. *European Economic Review*, 92, 110-132.](https://doi.org/10.1016/j.euroecorev.2016.11.009)
2. [Population-Weighted Regional Inequality Dataset (GitHub)](https://github.com/quarcs-lab/data-open/tree/master/pGDP)
3. [PyFixest Documentation](https://pyfixest.org/)
4. [Great Tables Documentation](https://posit-dev.github.io/great-tables/)